# Twitter Sentiment Classification: Positive - Neutral - Negative

In [1]:
import pandas as pd

df_train = pd.read_csv('twitter_sentiment_train.csv')
df_test  = pd.read_csv('twitter_sentiment_test.csv')

In [2]:
int_to_label = {2: 'Positive', 1: 'Neutral', 0: 'Negative'}

In [3]:
df_train.head(2)

,text,label
0,"""QT @user In the original draft of the 7th boo...",2
1,"""Ben Smith / Smith (concussion) remains out of...",1


In [4]:
y_train=df_train.pop('label')
y_test=df_test.pop('label')

#### Preprocessing Pipeline

In [5]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/giorgos/Documents/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/giorgos/Documents/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/giorgos/Documents/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [6]:
import re, string
from ekphrasis.classes.tokenizer import SocialTokenizer
tokenizer = SocialTokenizer(lowercase=False).tokenize

def uncontract(text):
    text = re.sub(r"(\b)([Aa]re|[Cc]ould|[Dd]id|[Dd]oes|[Dd]o|[Hh]ad|[Hh]as|[Hh]ave|[Ii]s|[Mm]ight|[Mm]ust|[Ss]hould|[Ww]ere|[Ww]ould)n't", r"\1\2 not", text)
    text = re.sub(r"(\b)([Hh]e|[Ii]|[Ss]he|[Tt]hey|[Ww]e|[Ww]hat|[Ww]ho|[Yy]ou)'ll", r"\1\2 will", text)
    text = re.sub(r"(\b)([Tt]hey|[Ww]e|[Ww]hat|[Ww]ho|[Yy]ou)'re", r"\1\2 are", text)
    text = re.sub(r"(\b)([Ii]|[Ss]hould|[Tt]hey|[Ww]e|[Ww]hat|[Ww]ho|[Ww]ould|[Yy]ou)'ve", r"\1\2 have", text)

    text = re.sub(r"(\b)([Cc]a)n't", r"\1\2n not", text)
    text = re.sub(r"(\b)([Ii])'m", r"\1\2 am", text)
    text = re.sub(r"(\b)([Ll]et)'s", r"\1\2 us", text)
    text = re.sub(r"(\b)([Ii]t)'s", r"\1\2 is", text)
    text = re.sub(r"(\b)([Tt]here)'s", r"\1\2 is", text)
    text = re.sub(r"(\b)([Ww])on't", r"\1\2ill not", text)
    text = re.sub(r"(\b)([Ss])han't", r"\1\2hall not", text)
    text = re.sub(r"(\b)([Yy])(?:'all|a'll)", r"\1\2ou all", text)

    return text


def tokenize_text(text):
    tokens = tokenizer(text)

    return tokens


def lowercase_tokens(tokens):
    tokens = [t.lower() for t in tokens]

    return tokens


def remove_stopwords(tokens):
    stop_words = stopwords.words('english')

    tokens = [t for t in tokens if t not in stop_words]

    return tokens


def lemmatize(tokens):
    lemmatizer = WordNetLemmatizer()

    lemmatized = [lemmatizer.lemmatize(t) for t in tokens]

    return lemmatized
 
def preprocessing_text(text):
   text = uncontract(text)
   text = tokenize_text(text)
   text = lowercase_tokens(text)
   text = remove_stopwords(text)
   text = lemmatize(text)

   return text

/home/giorgos/Documents/vscode/workspaces/epl499_group_project_team_2/.venv/lib/python3.13/site-packages/ekphrasis/classes/tokenizer.py:225: FutureWarning: Possible nested set at position 2190
  self.tok = re.compile(r"({})".format("|".join(pipeline)))


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
# Preprocess
clean_train_tweet_tokens=[tokenize_text(text) for text in df_train['text']]
clean_train_tweet_tokens=[lowercase_tokens(tokens) for tokens in clean_train_tweet_tokens]
clean_train_tweet_tokens=[remove_stopwords(tokens) for tokens in clean_train_tweet_tokens]
clean_train_tweet_tokens=[lemmatize(tokens) for tokens in clean_train_tweet_tokens]

clean_tweet_text_train = [' '.join(tokens) for tokens in clean_train_tweet_tokens]

# Train TF-IDF model on preprocessed training data
tfidf_vectorizer = TfidfVectorizer(
   ngram_range  = (1,1),
   max_features = 300,
   lowercase    = False,
   tokenizer    = None,
   preprocessor = None,
   stop_words   = None,
   min_df       = 10,
   max_df       = 0.80
)
tfidf_vectorizer.fit_transform(clean_tweet_text_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 218923 stored elements and shape (45615, 300)>

In [8]:
from sklearn.feature_extraction.text import CountVectorizer

def get_tfidf_matrix(texts: list[str], vectorizer: CountVectorizer) -> dict:
    """
    Takes a text string and a fitted CountVectorizer,
    and returns a dictionary with all n-grams in the vectorizer’s vocabulary as keys
    and their corresponding frequencies (counts) in the input text as values.

    Example:
    ----------
    Suppose your fitted vectorizer learned the following vocabulary:
        ["win", "best", "birthday gift", "please call"]

    Input text:
        "Please call now to win a gift! Please call again!"

    Returned dictionary:
        {
          "win": 1,
          "best": 0,
          "birthday gift": 0,
          "please call": 2
        }

    → Words or bigrams not found in the text have a count of 0.
    """
    # Apply preprocessing to text
    texts=[tokenize_text(text) for text in texts]
    texts=[lowercase_tokens(text) for text in texts]
    texts=[remove_stopwords(text) for text in texts]
    texts=[lemmatize(text) for text in texts]

    texts = [' '.join(text) for text in texts]
    
    tfidf_matrix = vectorizer.transform(texts)

    return tfidf_matrix

In [ ]:
from tqdm import tqdm
from feature_selection import *
X_train=df_train.copy()
X_test=df_test.copy()
"""
This block extracts multiple types of text-based features from the dataset (X_train and X_test)
by applying a list of feature-extraction functions one by one.

1. The list 'feature_functions' defines all feature extractors to be applied.
   Each function takes an email as input and returns dictionary of features (e.g., POS tag counts, n-gram frequencies).

2. The for-loop iterates through each feature extraction function with a progress bar (tqdm).

3. For each function:
      • Apply it to every email in X_train['text'], storing results as a list of dicts.
      • Convert the results to a temporary DataFrame (temp_df).
      • Reset indices and horizontally concatenate (pd.concat) the new features
        to X_train — effectively expanding the dataset with new columns.
      • Repeat the same process for X_test.

After this loop finishes:
      X_train and X_test contain both the original text column ('text')
      and all derived numeric and categorical features ready for modeling.
"""
feature_functions = [
log_number_of_tokens,
polarity_score,
subjectivity_score,
exclamation_count,
hashtag_count,
count_question,
mention_count,
has_url,
vader_scores,
count_happy_emoticons,
count_sad_emoticons,
count_negation,
negation_ratio,
emoji_sentiment,
count_elongated_words,
avg_token_length,
pos_tag_counts,
count_positive_words,
count_negative_words,
count_profanity
]

for func in tqdm(feature_functions):
    results = X_train['text'].apply(lambda x: func(str(x))).tolist()
    temp_df = pd.DataFrame(results)

    temp_df.reset_index(drop=True, inplace=True)
    X_train.reset_index(drop=True, inplace=True)
    X_train = pd.concat([X_train, temp_df], axis=1)

    results = X_test['text'].apply(lambda x: func(str(x))).tolist()
    temp_df = pd.DataFrame(results)

    temp_df.reset_index(drop=True, inplace=True)
    X_test.reset_index(drop=True, inplace=True)
    X_test = pd.concat([X_test, temp_df], axis=1)

texts = X_train['text'].astype(str).tolist()

tfidf_matrix = get_tfidf_matrix(X_train['text'], tfidf_vectorizer)
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)

X_train = pd.concat([X_train.reset_index(drop=True), tfidf_df], axis=1)

texts = X_test['text'].astype(str).tolist()

tfidf_matrix = get_tfidf_matrix(X_test['text'], tfidf_vectorizer)
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)

X_test = pd.concat([X_test.reset_index(drop=True), tfidf_df], axis=1)

/home/giorgos/Documents/vscode/workspaces/epl499_group_project_team_2/.venv/lib/python3.13/site-packages/ekphrasis/classes/tokenizer.py:225: FutureWarning: Possible nested set at position 2190
  self.tok = re.compile(r"({})".format("|".join(pipeline)))
 80%|████████  | 16/20 [01:01<00:13,  3.32s/it]

In [ ]:
X_train.head(10)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
X_train_dropped = X_train.drop(columns=["text"])
X_test_dropped = X_test.drop(columns=["text"])

model = RandomForestClassifier(n_estimators=100, random_state=123)
model.fit(X_train_dropped, y_train)

importances = model.feature_importances_

feature_importances = pd.DataFrame({
    'feature': X_train_dropped.columns,
    'importance': importances
})

sorted_feature_importances=feature_importances.sort_values(by="importance", ascending=False)

topk = sorted_feature_importances.head(50)

# Top k plot
plt.figure()
plt.barh(topk["feature"], topk["importance"])
# Highest value at the top
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top k by importance")

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD, Adam
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
import numpy as np

model_nn = Sequential([
    Dense(32, input_dim=X_train_dropped.shape[1], activation='relu'),   # hidden layer
    Dense(3, activation='softmax')                              # output layer
])

model_nn.compile(
    optimizer=SGD(learning_rate=0.01),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
model_nn.fit(X_train_dropped, y_train, epochs=20, batch_size=32, verbose=1)

# Predict
y_pred_prob_nn = model_nn.predict(X_test_dropped)
y_pred_nn = np.argmax(y_pred_prob_nn, axis=1) 

# Evaluate
print("Neural Network Accuracy:", accuracy_score(y_test, y_pred_nn))

In [ ]:
model_nn_topk = Sequential([
    Dense(32, input_dim=X_train_dropped[topk].shape[1], activation='relu'),   # hidden layer
    Dense(3, activation='softmax')                              # output layer
])

model_nn_topk.compile(
    optimizer=SGD(learning_rate=0.01),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
model_nn_topk.fit(X_train_dropped[topk], y_train, epochs=20, batch_size=32, verbose=1)

# Predict
y_pred_prob_nn = model_nn_topk.predict(X_test_dropped[topk])
y_pred_nn = np.argmax(y_pred_prob_nn, axis=1) 

# Evaluate
print("Neural Network Accuracy:", accuracy_score(y_test, y_pred_nn))